<a href="https://colab.research.google.com/github/Enricocmira/Codes/blob/main/Execu%C3%A7%C3%A3o_GSADF_em_segundo_plano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Execução em Segundo Plano Robusta (Método Recomendado)

Se o seu ambiente for desconectado ou você fechar o navegador, o processo normal do notebook pode parar. Para evitar isso e garantir que seus testes de bolhas rodem de forma ininterrupta, execute os passos abaixo:

In [ ]:
### Passo 1: Salvar o código do R em um arquivo script.R usando código nativo em R
script_content <- '
######################################################
## Código: Synchrony of Speculative Bubbles in      ##
##         Cryptocurrencies: Co-movement Dynamics   ##
##         and Behavioral Channels                  ##
######################################################

# Preparando o ambiente
rm(list=ls())
graphics.off()

# Instalando e carregando pacotes de forma robusta
packages <- c("tidyverse", "xts", "exuber", "quantmod", "e1071", "parallel", "doParallel", "readxl", "ggplot2")
installed_packages <- packages %in% rownames(installed.packages())
if (any(installed_packages == FALSE)) {
  install.packages(packages[!installed_packages], repos="https://cloud.r-project.org/")
}
invisible(lapply(packages, library, character.only = TRUE))

set.seed(123)

# Importar Dados
crypto_tickers <- c("BTC", "ETH", "LTC",  "XRP")
nboot = 20
lag_gsadf = 1
start_date <- as.Date("2018-01-01")
end_date <- as.Date("2026-09-30")

caminho_arquivo <- "/content/crypto_px_volume-1.xlsx"
df_excel <- read_excel(caminho_arquivo)

crypto_prices_wide <- df_excel %>%
  mutate(date = as.Date(date)) %>%
  filter(ticker %in% crypto_tickers,
         date >= start_date,
         date <= end_date) %>%
  dplyr::select(date, ticker, price) %>%
  pivot_wider(names_from = ticker, values_from = price) %>%
  arrange(date) %>%
  na.omit()

crypto_prices_log <- crypto_prices_wide
crypto_prices_log[,-1] <- log(crypto_prices_wide[,-1])

crypto_prices_df <- as.data.frame(crypto_prices_log)
colnames(crypto_prices_df)[1] <- "Date"

# Executando os testes em paralelo de forma otimizada
num_cores <- detectCores()
cl <- makeCluster(num_cores - 1)
registerDoParallel(cl)

print("Iniciando GSADF Univariado...")
panel_gsadf = crypto_prices_df %>% radf(lag = lag_gsadf)
cv_univariate = crypto_prices_df %>% radf_wb_cv2(nboot = nboot, adflag = lag_gsadf)

print("Iniciando GSADF Painel...")
cv_panel = crypto_prices_df %>%
  dplyr::select(-Date) %>% na.omit() %>%
  radf_sb_cv(lag = lag_gsadf, nboot = nboot)

stopCluster(cl)

attributes(panel_gsadf)$index = na.omit(crypto_prices_df)$Date
attributes(cv_panel)$index = na.omit(crypto_prices_df)$Date

save.image(file = "/content/workspace_completo_final.RData")
print("Sucesso total! Workspace final salvo em /content/workspace_completo_final.RData")
'

writeLines(script_content, "script.R")
cat("Script R gerado com sucesso!")

Script R gerado com sucesso!

In [ ]:
### Passo 2: Forçar o diretório /content e disparar a execução em background com system2
setwd("/content")
system2("/usr/bin/Rscript", args = c("/content/script.R"), stdout = "/content/execucao_r.log", stderr = "/content/execucao_r.log", wait = FALSE)

In [ ]:
### Passo 3: Monitorar o status do seu script de fundo usando R nativo
log_path <- "/content/execucao_r.log"
if (file.exists(log_path)) {
  log_content <- readLines(log_path, warn = FALSE)
  if (length(log_content) == 0) {
    cat("O arquivo de log foi criado, mas ainda está vazio. Aguarde alguns segundos...")
  } else {
    cat(paste(log_content, collapse = "\n"))
  }
} else {
  cat("O arquivo de log ainda não foi criado. Certifique-se de executar o Passo 2 primeiro.")
}

In [ ]:

# Verifica os processos ativos no sistema operacional que contêm 'Rscript'
sistema_processos <- system2("pgrep", args = "-fl Rscript", stdout = TRUE, stderr = TRUE)

if (length(sistema_processos) == 0) {
  cat("⚠️ O processo do Rscript NÃO está mais rodando no sistema.\n")
  cat("Dica: Verifique o final do arquivo de log (Passo 3) para ver se ele concluiu com sucesso ou se terminou com algum erro.\n")
} else {
  cat("✅ O script R ainda está rodando ativamente em segundo plano!\n\n")
  cat("Detalhes do processo:\n")
  print(sistema_processos)
}

✅ O script R ainda está rodando ativamente em segundo plano!

Detalhes do processo:
[1] "15901 sh"


In [ ]:
# Monitoramento e download automático utilizando R nativo
monitorar_e_baixar <- function(arquivo_esperado = "/content/workspace_completo_final.RData") {
  cat("⌛ Monitorando o processo do Rscript. Assim que terminar, iniciarei o download automaticamente...\n")

  repeat {
    # Verifica se o processo 'Rscript' ainda está ativo no sistema
    processos <- system2("pgrep", args = "-fl Rscript", stdout = TRUE, stderr = TRUE)

    if (length(processos) == 0) {
      cat("\n🛑 O processo Rscript parou de rodar!\n")
      break
    }

    # Mostra um indicador visual simples de progresso
    cat(".")
    utils::flush.console()
    Sys.sleep(15) # Aguarda 15 segundos antes de verificar novamente
  }

  # Verifica se o arquivo final foi criado e inicia o download usando o wrapper de JS do Colab no R
  if (file.exists(arquivo_esperado)) {
    cat(paste0("\n🎉 Arquivo localizado com sucesso! Iniciando download de: ", arquivo_esperado, "\n"))

    # Comando em JavaScript interpretado pelo Colab para download imediato do arquivo local
    js_code <- paste0("google.colab.files.download('", arquivo_esperado, "')")
    IRkernel::IRdisplay(IRkernel::javascript_display(js_code))
  } else {
    cat(paste0("\n❌ Atenção: O processo terminou, mas o arquivo ", arquivo_esperado, " não foi encontrado.\n"))
    cat("Dica: Execute o Passo 3 para visualizar o log e verificar se houve algum erro no script R.\n")
  }
}

# Executa a função de monitoramento
monitorar_e_baixar()

⌛ Monitorando o processo do Rscript. Assim que terminar, iniciarei o download automaticamente...
........................................................................................................................................................................................................................................................................................................................................................................